# 06 · Cross dataset generalization

Train on one dataset (or a combination) and test on another: twelve pairs, six models, five seeds.

- **Reads:** feature files, notebook 03 models
- **Writes:** `cross_dataset_results.csv`, `generalization_summary.csv`

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
# ============================================================
# NB06 Cell 1: Setup
# ============================================================

!pip install -q codecarbon

import os
import gc
import json
import time
import shutil
import tracemalloc
import tempfile
import warnings

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from codecarbon import EmissionsTracker

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")

SEEDS = [42, 52, 62, 72, 82]
DATASETS = ["FD001", "FD002", "FD003", "FD004"]
TARGET_COL = "failure_within_horizon"

WORKING_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_DIR / "NB-6-AGAIN-outputs"

PRED_DIR = OUTPUT_ROOT / "predictions"
RESULTS_DIR = OUTPUT_ROOT / "results"
META_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for p in [PRED_DIR, RESULTS_DIR, META_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Base input for dynamic searches
KAGGLE_INPUT = Path("/kaggle/input")

print("✅ NB06 Cell 1 completed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 7.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 71.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 77.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.2.0 requires cryptography<47,>=42, but you have cryptography 49.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have c

In [2]:
# ============================================================
# NB06 Cell 2: Load NB03 Results (Robust)
# ============================================================

print("🔍 Searching for NB03 summary files...")

centralized_paths = list(KAGGLE_INPUT.rglob("centralized_results.csv"))
threshold_paths = list(KAGGLE_INPUT.rglob("threshold_summary.csv"))

if not centralized_paths or not threshold_paths:
    raise FileNotFoundError(
        "❌ CRITICAL ERROR: Could not find 'centralized_results.csv' or 'threshold_summary.csv'.\n"
        "Please ensure NB-3-AGAIN-outputs is attached to this notebook."
    )

centralized_df = pd.read_csv(centralized_paths[0])
threshold_df = pd.read_csv(threshold_paths[0])

print(f"Centralized Results Shape: {centralized_df.shape}")
print(f"Threshold Summary Shape  : {threshold_df.shape}")

print("✅ NB06 Cell 2 completed")

🔍 Searching for NB03 summary files...
Centralized Results Shape: (120, 21)
Threshold Summary Shape  : (120, 4)
✅ NB06 Cell 2 completed


In [3]:
# ============================================================
# NB06 Cell 3: Dataset Pairs
# ============================================================

DATASET_PAIRS = []

for train_ds in DATASETS:
    for test_ds in DATASETS:
        if train_ds != test_ds:
            DATASET_PAIRS.append((train_ds, test_ds))

assert len(DATASET_PAIRS) == 12

print("Cross-Dataset Evaluation Pairs:")
for pair in DATASET_PAIRS:
    print(f"Train: {pair[0]} -> Test: {pair[1]}")

print("✅ NB06 Cell 3 completed")

Cross-Dataset Evaluation Pairs:
Train: FD001 -> Test: FD002
Train: FD001 -> Test: FD003
Train: FD001 -> Test: FD004
Train: FD002 -> Test: FD001
Train: FD002 -> Test: FD003
Train: FD002 -> Test: FD004
Train: FD003 -> Test: FD001
Train: FD003 -> Test: FD002
Train: FD003 -> Test: FD004
Train: FD004 -> Test: FD001
Train: FD004 -> Test: FD002
Train: FD004 -> Test: FD003
✅ NB06 Cell 3 completed


In [4]:
# ============================================================
# NB06 Cell 4: Loader
# ============================================================

INDEX_COLS = ["dataset", "engine_id", "cycle", "split"]

LABEL_COLS = [
    "RUL",
    "rul",
    "label_w10",
    "label_w20",
    "label_w30",
    "failure_within_horizon"
]

def load_test_dataset(dataset_name):
    search_target = f"{dataset_name}_test_features.parquet"
    found_paths = list(KAGGLE_INPUT.rglob(search_target))
    
    if not found_paths:
        raise FileNotFoundError(f"❌ Cannot find {search_target} in /kaggle/input/")
        
    df = pd.read_parquet(found_paths[0])
    df = df.loc[:, ~df.columns.duplicated()]

    feature_cols = [
        c for c in df.columns
        if c not in INDEX_COLS + LABEL_COLS
    ]

    assert len(feature_cols) == 342, f"Expected 342 features, got {len(feature_cols)}"

    return df, feature_cols

print("✅ NB06 Cell 4 completed")

✅ NB06 Cell 4 completed


In [5]:
# ============================================================
# NB06 Cell 5: Utilities & Model Loader
# ============================================================

# Model path cache for fast lookup inside the loop
_MODEL_PATH_CACHE = {}
_ALL_JOBLIB_PATHS = []

def load_model(dataset_name, model_name, seed):
    global _MODEL_PATH_CACHE, _ALL_JOBLIB_PATHS
    
    target_filename = f"{model_name}_seed{seed}.joblib"
    cache_key = f"{dataset_name}_{target_filename}"
    
    if cache_key in _MODEL_PATH_CACHE:
        return joblib.load(_MODEL_PATH_CACHE[cache_key])
        
    if not _ALL_JOBLIB_PATHS:
        _ALL_JOBLIB_PATHS = list(KAGGLE_INPUT.rglob("*.joblib"))
            
    valid_paths = [
        p for p in _ALL_JOBLIB_PATHS 
        if dataset_name.lower() in str(p).lower() and target_filename.lower() in str(p).lower()
    ]
    
    if not valid_paths:
        valid_paths = [p for p in _ALL_JOBLIB_PATHS if target_filename.lower() in str(p).lower()]

    if not valid_paths:
        raise FileNotFoundError(f"❌ CRITICAL ERROR: Cannot find model '{target_filename}' for '{dataset_name}'.")

    _MODEL_PATH_CACHE[cache_key] = valid_paths[0]
    return joblib.load(valid_paths[0])


def predict_proba_safe(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)


def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        joblib.dump(model, tmp.name)
        return os.path.getsize(tmp.name) / (1024 ** 2)


def parameter_count(model):
    try:
        if hasattr(model, "steps"):
            model = model.steps[-1][1]

        if hasattr(model, "calibrated_classifiers_"):
            model = getattr(model.calibrated_classifiers_[0], "estimator", getattr(model.calibrated_classifiers_[0], "base_estimator", model))
            if hasattr(model, "steps"): model = model.steps[-1][1]

        if hasattr(model, "coef_"):
            return getattr(model.coef_, "size", 0) + getattr(model.intercept_, "size", 0)

        if hasattr(model, "estimators_"):
            return sum(getattr(est.tree_, "node_count", 1) for est in model.estimators_)

        if hasattr(model, "tree_count_"):
            return model.tree_count_

        if hasattr(model, "get_booster"):
            return len(model.get_booster().get_dump())
            
        if hasattr(model, "booster_"):
            return model.booster_.num_trees()
    except:
        pass
    return 1


def estimate_flops(model, n_features):
    return max(parameter_count(model) * n_features * 2, 1)

print("✅ NB06 Cell 5 completed")

✅ NB06 Cell 5 completed


In [6]:
# ============================================================
# NB06 Cell 6: Main Loop
# ============================================================

all_results = []
deployment_results = []

total_runs = 12 * 6 * 5
run_id = 0

for train_ds, test_ds in DATASET_PAIRS:
    test_df, FEATURE_COLS = load_test_dataset(test_ds)
    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COL]

    for model_name in ["logistic_regression", "random_forest", "xgboost", "lightgbm", "catboost", "svm"]:
        for seed in SEEDS:
            run_id += 1
            print(f"[{run_id}/{total_runs}] {train_ds} → {test_ds} | {model_name} | seed={seed}")

            # Load model dynamically
            model = load_model(train_ds, model_name, seed)

            # Get source threshold
            threshold = threshold_df[
                (threshold_df["dataset"] == train_ds) &
                (threshold_df["model"] == model_name) &
                (threshold_df["seed"] == seed)
            ]["threshold"].iloc[0]

            # Get source baseline performance
            source_row = centralized_df[
                (centralized_df["dataset"] == train_ds) &
                (centralized_df["model"] == model_name) &
                (centralized_df["seed"] == seed)
            ]
            source_f1 = source_row["f1"].iloc[0]
            source_auc = source_row["roc_auc"].iloc[0]

            tracemalloc.start()
            tracker = EmissionsTracker(project_name="NB06", save_to_file=False, allow_multiple_runs=True, log_level="error")
            tracker.start()
            
            start = time.time()
            probs = predict_proba_safe(model, X_test)
            inference_time = time.time() - start
            
            energy_kwh = tracker.stop() or 0.0
            _, peak_mem = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            preds = (probs >= threshold).astype(int)

            accuracy = accuracy_score(y_test, preds)
            precision = precision_score(y_test, preds, zero_division=0)
            recall = recall_score(y_test, preds, zero_division=0)
            f1 = f1_score(y_test, preds, zero_division=0)
            auc = roc_auc_score(y_test, probs)

            latency_ms = (inference_time / len(X_test)) * 1000
            throughput_sps = (len(X_test) / inference_time)
            memory_mb = (peak_mem / (1024 ** 2))

            model_size_mb = get_model_size_mb(model)
            params = parameter_count(model)
            flops = estimate_flops(model, len(FEATURE_COLS))

            gap_f1 = source_f1 - f1
            gap_auc = source_auc - auc

            all_results.append({
                "train_dataset": train_ds,
                "test_dataset": test_ds,
                "model": model_name,
                "seed": seed,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc_auc": auc,
                "generalization_gap_f1": gap_f1,
                "generalization_gap_auc": gap_auc,
                "inference_time_sec": inference_time,
                "latency_ms": latency_ms,
                "throughput_sps": throughput_sps,
                "memory_mb_inference": memory_mb,
                "model_size_mb": model_size_mb,
                "parameter_count": params,
                "flops_estimated": flops,
                "energy_kwh": energy_kwh
            })

            pred_df = pd.DataFrame({
                "engine_id": test_df["engine_id"],
                "cycle": test_df["cycle"],
                "y_true": y_test,
                "probability": probs,
                "prediction": preds
            })

            pred_path = PRED_DIR / f"{train_ds}_to_{test_ds}_{model_name}_seed{seed}.csv"
            pred_df.to_csv(pred_path, index=False)

            gc.collect()

print("✅ NB06 Cell 6 completed")

[1/360] FD001 → FD002 | logistic_regression | seed=42


[codecarbon WARNING @ 18:35:10] Multiple instances of codecarbon are allowed to run at the same time.


[2/360] FD001 → FD002 | logistic_regression | seed=52
[3/360] FD001 → FD002 | logistic_regression | seed=62
[4/360] FD001 → FD002 | logistic_regression | seed=72
[5/360] FD001 → FD002 | logistic_regression | seed=82
[6/360] FD001 → FD002 | random_forest | seed=42
[7/360] FD001 → FD002 | random_forest | seed=52
[8/360] FD001 → FD002 | random_forest | seed=62
[9/360] FD001 → FD002 | random_forest | seed=72
[10/360] FD001 → FD002 | random_forest | seed=82
[11/360] FD001 → FD002 | xgboost | seed=42
[12/360] FD001 → FD002 | xgboost | seed=52
[13/360] FD001 → FD002 | xgboost | seed=62
[14/360] FD001 → FD002 | xgboost | seed=72
[15/360] FD001 → FD002 | xgboost | seed=82
[16/360] FD001 → FD002 | lightgbm | seed=42
[17/360] FD001 → FD002 | lightgbm | seed=52
[18/360] FD001 → FD002 | lightgbm | seed=62
[19/360] FD001 → FD002 | lightgbm | seed=72
[20/360] FD001 → FD002 | lightgbm | seed=82
[21/360] FD001 → FD002 | catboost | seed=42
[22/360] FD001 → FD002 | catboost | seed=52
[23/360] FD001 → FD0

In [7]:
# ============================================================
# NB06 Cell 7: Save Results
# ============================================================

results_df = pd.DataFrame(all_results)

deployment_df = results_df[[
    "train_dataset",
    "test_dataset",
    "model",
    "seed",
    "latency_ms",
    "throughput_sps",
    "memory_mb_inference",
    "model_size_mb",
    "parameter_count",
    "flops_estimated",
    "energy_kwh"
]]

summary_df = (
    results_df
    .groupby(["model"])
    .agg({
        "f1": ["mean", "std"],
        "roc_auc": ["mean", "std"],
        "generalization_gap_f1": "mean",
        "generalization_gap_auc": "mean"
    })
)

results_df.to_csv(RESULTS_DIR / "cross_dataset_results.csv", index=False)
deployment_df.to_csv(RESULTS_DIR / "deployment_metrics.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "generalization_summary.csv")

print("✅ NB06 Cell 7 completed")

✅ NB06 Cell 7 completed


In [8]:
# ============================================================
# NB06 Cell 8: Validation + ZIP
# ============================================================

assert len(results_df) == 360, f"Expected 360 rows, got {len(results_df)}"

assert (results_df["throughput_sps"] > 0).all(), "Invalid throughput values found"
assert (results_df["memory_mb_inference"] > 0).all(), "Invalid memory inference values found"
assert (results_df["parameter_count"] > 0).all(), "Invalid parameter count values found"
assert (results_df["flops_estimated"] > 0).all(), "Invalid flops estimated values found"

with open(META_DIR / "run_config.json", "w") as f:
    json.dump({
        "datasets": DATASETS,
        "pairs": DATASET_PAIRS,
        "seeds": SEEDS
    }, f, indent=2)

zip_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    root_dir=str(OUTPUT_ROOT)
)

print("=" * 80)
print("✅ NB06 COMPLETED SUCCESSFULLY")
print(f"Rows: {len(results_df)}")
print(f"Predictions: {len(list(PRED_DIR.glob('*.csv')))}")
print(f"ZIP: {zip_path}")
print("=" * 80)

✅ NB06 COMPLETED SUCCESSFULLY
Rows: 360
Predictions: 360
ZIP: /kaggle/working/NB-6-AGAIN-outputs.zip
